In [ ]:
import numpy as np
import pandas as pd
from mechaphlowers import SectionArray, PlotEngine, SectionStudy
import plotly.graph_objects as go

from mechaphlowers.data.catalog.catalog import sample_cable_catalog


## 1. Define input objects

In [ ]:
section_array = SectionArray(
    pd.DataFrame(
        {
            "name": ["1", "2", "3", "4"],
            "suspension": [False, True, True, False],
            "conductor_attachment_altitude": [30, 50, 60, 65],
            "crossarm_length": [0, 10, 10, 0],
            "line_angle": [0, 10, 0, 0],
            "insulator_length": [3, 3, 3, 3],
            "span_length": [500, 300, 400, np.nan],
            "insulator_mass": [100, 50, 50, 100],
            "load_mass": [0, 0, 0, 0],
            "load_position": [0, 0, 0, 0],
            "counterweight_mass": [0, 0, 0, 0],
        }
    ),
    sagging_parameter=2000,
    sagging_temperature=15
)
#specify that angles are in degrees
section_array.add_units({"line_angle": "deg"})


cable_array = sample_cable_catalog.get_as_object(["ASTER600"])

## 2. Create and run BalanceEngine

In [ ]:


study = SectionStudy(cable_array, section_array)

# always run solve_adjustement after creating BalanceEngine object
study.solve_adjustment()
study.solve_change_state(new_temperature=30)

## 3. Plot the graph

In [ ]:
# Plot in 3D

plot_engine = PlotEngine(study.balance_engine)
fig = go.Figure()

plot_engine.preview_line3d(fig)
fig.show()

In [ ]:
# Add wind and then plot

study.solve_change_state(wind_pressure=300)
plot_engine = PlotEngine(study.balance_engine)
fig_wind = go.Figure()

plot_engine.preview_line3d(fig_wind)
fig_wind.show()

# 4. Obstacles and distances

In [ ]:
study.solve_change_state(wind_pressure=300)
plot_engine = PlotEngine(study.balance_engine)
plot_engine.position_engine.add_obstacle(
        name="obstacle_0",
        span_index=1,
        coords=np.array([[100, -10, 30]]),
        support_reference='left',
    )

fig_distances = go.Figure()
plot_engine.point_distance(
    fig=fig_distances,
    obstacle_name="obstacle_0",
    point_index=0,
)

plot_engine.preview_line3d(fig_distances)
fig_distances.show()